# Module 5: Risk Analysis

**QuantVerse** — Quantitative Portfolio Intelligence System

---

## Objectives

Comprehensive risk analysis for all optimized portfolios from Module 4:

1. **VaR/CVaR** — Parametric, Historical, Cornish-Fisher, Monte Carlo
2. **Drawdown Analysis** — Maximum drawdown, underwater curves, Calmar/Sterling ratios
3. **Tail Risk** — Extreme Value Theory (GPD), Hill estimator, tail concentration
4. **Factor Risk Decomposition** — Marginal risk contribution, PCA factors, asset class attribution
5. **Risk Concentration** — Diversification ratio, effective number of bets

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, logging, sys, os, json

sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
sns.set_palette('husl')
print('Setup complete.')

In [ ]:
# Load data
data_dir = '../data/processed'
daily_returns = pd.read_parquet(f'{data_dir}/returns_daily.parquet')
clean_prices = pd.read_parquet(f'{data_dir}/prices_clean.parquet')

with open(f'{data_dir}/asset_class_map.json', 'r') as f:
    class_map = json.load(f)

# Investable assets
signal_tickers = [t for t, c in class_map.items() if c == 'signals']
investable = [t for t in daily_returns.columns if t not in signal_tickers]
returns = daily_returns[investable].dropna()

# Load portfolio weights from Module 4
weights_path = f'{data_dir}/portfolio_weights.parquet'
if os.path.exists(weights_path):
    all_weights = pd.read_parquet(weights_path)
    print(f'Loaded {all_weights.shape[1]} portfolio strategies from Module 4')
else:
    # Fallback: equal weight
    all_weights = pd.DataFrame({'Equal Weight': pd.Series(1/len(investable), index=investable)})
    print('Module 4 weights not found — using equal weight')

print(f'Assets: {len(investable)}, Obs: {len(returns)}')
print(f'Strategies: {list(all_weights.columns)}')

## 1. VaR/CVaR Comparison Across Methods

Four approaches to estimate tail loss:
- **Parametric (Gaussian)**: assumes normality — underestimates for fat tails
- **Historical**: empirical quantile — no assumptions, limited by sample size
- **Cornish-Fisher**: adjusts Gaussian for skewness/kurtosis
- **Monte Carlo**: simulation-based — flexible but depends on assumed distribution

In [ ]:
from project.risk import VaRCVaRCalculator

# Pick a primary strategy (Max Sharpe or first available)
primary = 'Max Sharpe' if 'Max Sharpe' in all_weights.columns else all_weights.columns[0]
w_primary = all_weights[primary]

var_calc = VaRCVaRCalculator(returns, weights=w_primary)
comparison = var_calc.compare_all(alpha=0.05)

print(f'VaR/CVaR Comparison — {primary} Portfolio (α = 5%)')
print('=' * 80)
print((comparison * 100).round(4).to_string())
print('\n(All values are daily percentage points — do NOT scale by √252 for annual CVaR;')
print(' CVaR annualization requires full simulation, not variance-based scaling)')

In [ ]:
# VaR/CVaR for ALL strategies
strategy_risk = {}
for strat in all_weights.columns:
    calc = VaRCVaRCalculator(returns, weights=all_weights[strat])
    hist = calc.historical(alpha=0.05)
    cf = calc.cornish_fisher(alpha=0.05)
    strategy_risk[strat] = {
        'VaR_Hist_%': hist['VaR'] * 100,
        'CVaR_Hist_%': hist['CVaR'] * 100,
        'VaR_CF_%': cf['VaR'] * 100,
        'CVaR_CF_%': cf['CVaR'] * 100,
    }

risk_df = pd.DataFrame(strategy_risk).T
print('Strategy Risk Comparison (Daily, 5%)')
print('=' * 70)
print(risk_df.round(4).to_string())

In [ ]:
# Visualize VaR/CVaR across strategies
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

risk_df[['VaR_Hist_%', 'VaR_CF_%']].plot(kind='barh', ax=axes[0],
    color=['steelblue', 'coral'], edgecolor='white')
axes[0].set_title('Daily VaR (5%) by Strategy', fontweight='bold')
axes[0].set_xlabel('VaR (%)')

risk_df[['CVaR_Hist_%', 'CVaR_CF_%']].plot(kind='barh', ax=axes[1],
    color=['steelblue', 'coral'], edgecolor='white')
axes[1].set_title('Daily CVaR (5%) by Strategy', fontweight='bold')
axes[1].set_xlabel('CVaR (%)')

plt.tight_layout()
plt.show()

In [ ]:
# Component VaR decomposition
comp_var = var_calc.component_var(alpha=0.05)

fig, ax = plt.subplots(figsize=(14, 7))
top15 = comp_var.head(15)
colors = ['coral' if v > 0 else 'steelblue' for v in top15['Pct_Contribution']]
top15['Pct_Contribution'].plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_xlabel('% of Portfolio VaR')
ax.set_title(f'Component VaR — Top 15 Risk Contributors ({primary})',
             fontsize=13, fontweight='bold')
ax.axvline(x=0, color='gray', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Rolling VaR/CVaR
rolling = var_calc.rolling_var(window=252, alpha=0.05)

fig, ax = plt.subplots(figsize=(16, 6))
ax.fill_between(rolling.index, rolling['CVaR'] * 100, alpha=0.3, color='red', label='CVaR (5%)')
ax.plot(rolling.index, rolling['VaR'] * 100, color='darkred', lw=1.5, label='VaR (5%)')
ax.set_ylabel('Daily Loss (%)')
ax.set_title(f'Rolling 1-Year VaR & CVaR — {primary} Portfolio', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Drawdown Analysis

In [ ]:
from project.risk import DrawdownAnalyzer

dd = DrawdownAnalyzer(returns, weights=w_primary)
dd_summary = dd.summary()

print(f'Drawdown Summary — {primary} Portfolio')
print('=' * 50)
for k, v in dd_summary.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

In [ ]:
# Top drawdown episodes
top_dd = dd.top_drawdowns(n=10)
print('Top 10 Drawdown Episodes:')
print(top_dd.to_string())

In [ ]:
# Underwater plot
uw = dd.underwater_data()

fig, axes = plt.subplots(2, 1, figsize=(16, 10), gridspec_kw={'height_ratios': [2, 1]}, sharex=True)

# Cumulative returns
axes[0].plot(uw.index, uw['Cumulative'], color='steelblue', lw=1.5, label='Portfolio')
axes[0].plot(uw.index, uw['Peak'], color='gray', lw=0.8, linestyle='--', alpha=0.7, label='Peak')
axes[0].set_ylabel('Cumulative Return')
axes[0].set_title(f'Equity Curve — {primary} Portfolio', fontsize=14, fontweight='bold')
axes[0].legend()

# Underwater
axes[1].fill_between(uw.index, uw['Underwater'] * 100, 0, color='red', alpha=0.4)
axes[1].set_ylabel('Drawdown (%)')
axes[1].set_title('Underwater Curve', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Drawdown comparison across strategies
dd_comparison = {}
for strat in all_weights.columns:
    dd_strat = DrawdownAnalyzer(returns, weights=all_weights[strat])
    s = dd_strat.summary()
    dd_comparison[strat] = {
        'Max_DD_%': s['Max_Drawdown_%'],
        'Current_DD_%': s['Current_Drawdown_%'],
        'Calmar': s['Calmar_Ratio'],
        'Sterling': s['Sterling_Ratio'],
        'Ulcer_Index': s['Ulcer_Index'],
    }

dd_comp_df = pd.DataFrame(dd_comparison).T
print('Drawdown Metrics by Strategy:')
print('=' * 80)
print(dd_comp_df.round(4).to_string())

In [ ]:
# Multi-strategy underwater comparison
fig, ax = plt.subplots(figsize=(16, 7))

strategies_to_plot = list(all_weights.columns)[:6]
colors = plt.cm.Set2(np.linspace(0, 1, len(strategies_to_plot)))

for idx, strat in enumerate(strategies_to_plot):
    dd_s = DrawdownAnalyzer(returns, weights=all_weights[strat])
    dd_ts = dd_s.portfolio_drawdown()
    ax.plot(dd_ts.index, dd_ts * 100, color=colors[idx], lw=1.2, alpha=0.8, label=strat)

ax.set_ylabel('Drawdown (%)')
ax.set_title('Drawdown Comparison Across Strategies', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='lower left')
plt.tight_layout()
plt.show()

## 3. Tail Risk — Extreme Value Theory

In [ ]:
from project.risk import TailRiskAnalyzer

port_returns = pd.Series(returns.values @ w_primary.reindex(investable).fillna(0).values,
                         index=returns.index)
tail = TailRiskAnalyzer(port_returns)

# GPD fit to left tail
gpd = tail.fit_gpd_tail(threshold_pct=5.0, tail='left')
print(f'GPD Tail Fit — {primary} Portfolio')
print('=' * 60)
print(f"  Tail index (ξ):    {gpd['shape_xi']:.4f}")
print(f"  Scale (σ):         {gpd['scale_sigma']:.6f}")
print(f"  KS p-value:        {gpd['KS_pvalue']:.4f}")
print(f"  N exceedances:     {gpd['n_exceedances']}")
print(f"  EVT VaR (1%):      {gpd['VaR_1pct']*100:.4f}%")
print(f"  EVT CVaR (1%):     {gpd['CVaR_1pct']*100:.4f}%")
print(f"  EVT VaR (5%):      {gpd['VaR_5pct']*100:.4f}%")
print(f"  EVT CVaR (5%):     {gpd['CVaR_5pct']*100:.4f}%")
print(f"\n  ξ > 0 → heavy tail (power law), ξ = 0 → exponential, ξ < 0 → bounded")

In [ ]:
# Expected Shortfall at multiple confidence levels
es = tail.expected_shortfall_table()
print('Expected Shortfall Table:')
print('=' * 90)
print((es.drop(columns=['Alpha'])).to_string())

In [ ]:
# Hill estimator — tail index stability plot
hill = tail.hill_estimator()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(hill['k'], hill['alpha'], color='steelblue', lw=1.2)
ax.axhline(y=4, color='red', linestyle='--', alpha=0.5, label='α=4 (finite kurtosis)')
ax.axhline(y=2, color='darkred', linestyle='--', alpha=0.5, label='α=2 (finite variance)')
ax.set_xlabel('k (number of order statistics)')
ax.set_ylabel('Tail index α')
ax.set_title('Hill Estimator — Tail Index Stability Plot', fontsize=13, fontweight='bold')
ax.legend()
ax.set_ylim(0, 10)
plt.tight_layout()
plt.show()

In [ ]:
# Tail concentration
tc = tail.tail_concentration()
print('Tail Concentration — What fraction of total losses comes from extreme events?')
print('=' * 75)
print(tc.round(4).to_string(index=False))

In [ ]:
# Return distribution with VaR/CVaR markers
fig, ax = plt.subplots(figsize=(14, 6))

ax.hist(port_returns * 100, bins=100, density=True, alpha=0.6, color='steelblue',
        edgecolor='white', label='Portfolio Returns')

hist_var = var_calc.historical(0.05)
hist_cvar = hist_var['CVaR']
ax.axvline(-hist_var['VaR'] * 100, color='orange', lw=2, linestyle='--',
           label=f"VaR 5%: {hist_var['VaR']*100:.2f}%")
ax.axvline(-hist_cvar * 100, color='red', lw=2, linestyle='--',
           label=f"CVaR 5%: {hist_cvar*100:.2f}%")

ax.set_xlabel('Daily Return (%)')
ax.set_ylabel('Density')
ax.set_title(f'Return Distribution with Risk Markers — {primary}',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 4. Factor Risk Decomposition

In [ ]:
from project.risk import FactorRiskDecomposer

frd = FactorRiskDecomposer(returns, weights=w_primary, asset_class_map=class_map)

# Marginal risk contribution
mrc = frd.marginal_risk_contribution()
print(f'Top 15 Risk Contributors — {primary}')
print('=' * 80)
print(mrc.head(15)[['Weight_%', 'Pct_of_Risk', 'Asset_Class']].round(2).to_string())

In [ ]:
# Asset class risk attribution
ac_risk = frd.asset_class_risk()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ac_risk[['Weight_%', 'Pct_of_Risk']].plot(kind='barh', ax=axes[0],
    color=['steelblue', 'coral'], edgecolor='white')
axes[0].set_xlabel('%')
axes[0].set_title('Weight vs Risk Contribution by Asset Class', fontweight='bold')

ac_risk['Risk_Weight_Ratio'].plot(kind='barh', ax=axes[1],
    color='mediumpurple', edgecolor='white')
axes[1].axvline(x=1, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Risk/Weight Ratio')
axes[1].set_title('Risk Efficiency (>1 = risk overweight)', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# PCA factor decomposition
pca_result = frd.pca_factor_decomposition(n_factors=5)

print('PCA Factor Risk Decomposition:')
print('=' * 60)
print(f"  R² (factors explain):    {pca_result['r_squared']:.4f}")
print(f"  Systematic risk:         {pca_result['systematic_risk_%']:.1f}%")
print(f"  Idiosyncratic risk:      {pca_result['idiosyncratic_risk_%']:.1f}%")
print(f"  Total vol (annual):      {pca_result['total_vol_annual']*100:.2f}%")
print(f"  Systematic vol (annual): {pca_result['systematic_vol_annual']*100:.2f}%")
print(f"\nFactor Contributions:")
print(pca_result['factor_contributions'].round(4).to_string())

In [ ]:
# Risk concentration comparison across strategies
conc_comparison = {}
for strat in all_weights.columns:
    frd_s = FactorRiskDecomposer(returns, weights=all_weights[strat], asset_class_map=class_map)
    conc = frd_s.concentration_metrics()
    conc_comparison[strat] = conc

conc_df = pd.DataFrame(conc_comparison).T
print('Risk Concentration Metrics by Strategy:')
print('=' * 90)
print(conc_df.round(4).to_string())

In [ ]:
# Diversification ratio vs max drawdown
fig, ax = plt.subplots(figsize=(10, 7))

for strat in all_weights.columns:
    dr = conc_df.loc[strat, 'Diversification_Ratio']
    mdd = dd_comp_df.loc[strat, 'Max_DD_%']
    ax.scatter(dr, abs(mdd), s=150, zorder=5, edgecolors='white', linewidth=2)
    ax.annotate(strat, (dr, abs(mdd)), fontsize=9, textcoords='offset points',
                xytext=(5, 5))

ax.set_xlabel('Diversification Ratio', fontsize=13)
ax.set_ylabel('Max Drawdown (%)', fontsize=13)
ax.set_title('Diversification vs Maximum Drawdown', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Drawdown-at-Risk

In [ ]:
dar = dd.drawdown_at_risk(alpha=0.05)
print(f'Drawdown-at-Risk (5%) — {primary}')
print(f"  DaR:  {dar['DaR_%']:.2f}%")
print(f"  CDaR: {dar['CDaR_%']:.2f}%")
print(f"\n  DaR = worst 5th percentile drawdown")
print(f"  CDaR = average drawdown given drawdown ≥ DaR")

## 6. Comprehensive Risk Dashboard

In [ ]:
# Build master risk table
master_risk = {}
for strat in all_weights.columns:
    w = all_weights[strat]
    
    # VaR/CVaR
    calc = VaRCVaRCalculator(returns, weights=w)
    h = calc.historical(0.05)
    
    # Drawdown
    dd_s = DrawdownAnalyzer(returns, weights=w)
    s = dd_s.summary()
    
    # Concentration
    frd_s = FactorRiskDecomposer(returns, weights=w, asset_class_map=class_map)
    c = frd_s.concentration_metrics()
    
    master_risk[strat] = {
        'Ann_Vol_%': s['Ann_Volatility_%'],
        'VaR_5%': h['VaR'] * 100,
        'CVaR_5%': h['CVaR'] * 100,
        'Max_DD_%': s['Max_Drawdown_%'],
        'Calmar': s['Calmar_Ratio'],
        'Ulcer_Index': s['Ulcer_Index'],
        'Div_Ratio': c['Diversification_Ratio'],
        'ENB_Risk': c['ENB_Risk'],
    }

master_df = pd.DataFrame(master_risk).T
print('Master Risk Dashboard')
print('=' * 100)
print(master_df.round(4).to_string())

# Export for Module 9
master_df.to_parquet(f'{data_dir}/risk_metrics.parquet')
print('Risk metrics exported.')

In [ ]:
# Export risk data for Module 6
master_df.to_parquet(f'{data_dir}/risk_metrics.parquet')
print(f'Risk metrics exported to {data_dir}/risk_metrics.parquet')

---

## Key Takeaways from Module 5

1. **Gaussian VaR underestimates** tail risk — Cornish-Fisher and Historical always show higher losses
2. **GPD tail fitting** reveals the true shape of extreme losses — ξ > 0 confirms power-law tails
3. **Tail concentration** is high — a small fraction of worst days accounts for the majority of total losses
4. **Drawdown profiles differ significantly** across strategies — HRP and Risk Parity tend to have shallower drawdowns
5. **Risk contributions ≠ weight allocations** — crypto assets contribute disproportionate risk relative to their weight
6. **Diversification ratio** is a strong predictor of drawdown resilience

### Next: Module 6 — Monte Carlo Simulation & Stress Testing

Forward 12-month simulations, scenario analysis, and portfolio stress testing.